## Background
This notebook creates prompts for further Gemma 2 fine-tuning un casual language modeling setting.

## Imports

In [ ]:
import warnings
import ast
import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import joblib

In [ ]:
warnings.filterwarnings("ignore")

## Constants

In [ ]:
TRAIN_PATH = "../../data/"
TRAIN_NAME = "train.parquet"
TEST_NAME = "test.csv"

## Read data

In [ ]:
train = pd.read_parquet(os.path.join(TRAIN_PATH, TRAIN_NAME))
display(train.head())

test = pd.read_csv(os.path.join(TRAIN_PATH, TEST_NAME))
display(test.head())

,id,content,lang,manipulative,techniques,trigger_words
0,0bb0c7fa-101b-4583-a5f9-9d503339141c,Новий огляд мапи DeepState від російського вій...,uk,True,"[euphoria, loaded_language]","[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,Недавно 95 квартал жёстко поглумился над русск...,ru,True,"[loaded_language, cherry_picking]","[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,e6a427f1-211f-405f-bd8b-70798458d656,🤩\nТим часом йде евакуація Бєлгородського авто...,uk,True,"[loaded_language, euphoria]","[[55, 100]]"
3,1647a352-4cd3-40f6-bfa1-d87d42e34eea,В Україні найближчим часом мають намір посилит...,uk,False,None,None
4,9c01de00-841f-4b50-9407-104e9ffb03bf,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",ru,True,[loaded_language],"[[114, 144]]"


,id,content,techniques,trigger_words
0,521cd2e8-dd9f-42c4-98ba-c0c8890ff1ba,"Они просрали нашу технику, положили кучу людей...","['fud', 'loaded_language']","[(0, 12), (27, 46), (48, 71), (131, 162), (164..."
1,9b2a61e4-d14e-4ff7-b304-e73d720319bf,❗️\nКитай предлагает отдать оккупированные тер...,['loaded_language'],"[(374, 425)]"
2,f0f1c236-80a8-4d25-b30c-a420a39be632,Сегодня будет ровно 6 месяцев с этого обещания...,['loaded_language'],"[(0, 127)]"
3,31ea05ba-2c2b-4b84-aba7-f3cf6841b204,⚡️\nІзраїль вперше у світі збив балістичну рак...,[],NaN
4,a79e13ec-6d9a-40b5-b54c-7f4f743a7525,Склав невелику навчально-методичну таблицю на ...,['loaded_language'],"[(87, 103), (127, 136), (170, 189), (204, 255)..."


In [207]:
# Extract trigger_words phrases from the text:
trigger_words_phrases = []
for text, trigger_words in zip(train.content.values, train.trigger_words.values):
    if text is None or trigger_words is None:
        trigger_words_phrases.append([])
        continue
    trigger_words_phrases.append([text[i:j] for i, j in trigger_words])
train["trigger_words_phrases"] = trigger_words_phrases

In [80]:
from collections import Counter
unique_techniques = Counter()
for t in train["techniques"]:
    unique_techniques.update(t)
techniques_dict = {k: i for i, k in enumerate(unique_techniques.keys())}
techniques_dict

{'euphoria': 0,
 'loaded_language': 1,
 'cherry_picking': 2,
 'glittering_generalities': 3,
 'cliche': 4,
 'appeal_to_fear': 5,
 'bandwagon': 6,
 'fud': 7,
 'whataboutism': 8,
 'straw_man': 9}

## Get text embeddings

In [ ]:
# Get vectors for texts:
model = SentenceTransformer('Alibaba-NLP/gte-multilingual-base', trust_remote_code=True, device="mps")

Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: {'classifier.bias', 'classifier.weight'}
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
all_texts_to_encode = train["content"].to_list() + train["trigger_words_phrases"].apply(lambda x: ", ".join(x)).to_list() + test["content"].to_list()
all_texts_to_encode = [t[:5000] for t in all_texts_to_encode]

text_vectors_all = model.encode(all_texts_to_encode, show_progress_bar=True, normalize_embeddings=True, batch_size=8)
text_vectors = text_vectors_all[:len(train)]
triger_phrases_vectors = text_vectors_all[len(train):-len(test)]
text_vectors_test = text_vectors_all[-len(test):]

Batches: 100%|██████████| 1673/1673 [04:30<00:00,  6.19it/s]


## Get examples for few-shot prompts

In [ ]:
# For each sample id get 3 most similar texts and their techniques:
# (3 most similar by text and 3 most similar by trigger phrases)
# Avoid elanples longer than 500 characters


N = 2

# Text_ids to 0 (longer than 500 characters):
long_texts_ids = []
for i in range(len(train)):
    if len(train["content"].iloc[i]) > 500:
        long_texts_ids.append(i)

sim_texts, sim_measur, sim_techniques, trigger_phrases = [], [], [], []
for i in tqdm(range(len(train))):
    sample_vector = text_vectors[i]
    similarities = cosine_similarity(sample_vector.reshape(1, -1), triger_phrases_vectors)[0]
    # Make sure that the sample is not in the top:
    similarities[i] = 0
    # Examples longer than 500 characters are not considered to be similar:
    for j in long_texts_ids:
        similarities[j] = 0
    most_similar_indices = similarities.argsort()[-N:][::-1]
    sim_texts.append([train["content"].iloc[j] for j in most_similar_indices])
    sim_measur.append([similarities[j] for j in most_similar_indices])
    sim_techniques.append([train["techniques"].iloc[j] for j in most_similar_indices])
    trigger_phrases.append([train["trigger_words_phrases"].iloc[j] for j in most_similar_indices])

# The same but text_vectors instead of triger_phrases_vectors:
sim_texts_t, sim_measur_t, sim_techniques_t, trigger_phrases_t = [], [], [], []
for i in tqdm(range(len(train))):
    sample_vector = text_vectors[i]
    similarities = cosine_similarity(sample_vector.reshape(1, -1), text_vectors)[0]
    # Make sure that the sample is not in the top:
    similarities[i] = 0
    for j in long_texts_ids:
        similarities[j] = 0
    most_similar_indices = similarities.argsort()[-N:][::-1]
    sim_texts_t.append([train["content"].iloc[j] for j in most_similar_indices])
    sim_measur_t.append([similarities[j] for j in most_similar_indices])
    sim_techniques_t.append([train["techniques"].iloc[j] for j in most_similar_indices])
    trigger_phrases_t.append([train["trigger_words_phrases"].iloc[j] for j in most_similar_indices])

100%|██████████| 3822/3822 [00:13<00:00, 292.62it/s]


In [233]:
# Building the same for test set:
sim_texts_test, sim_measur_test, sim_techniques_test, trigger_phrases_test = [], [], [], []
for i in tqdm(range(len(test))):
    sample_vector = text_vectors_test[i]
    similarities = cosine_similarity(sample_vector.reshape(1, -1), triger_phrases_vectors)[0]
    # Examples longer than 500 characters are not considered to be similar:
    for j in long_texts_ids:
        similarities[j] = 0
    most_similar_indices = similarities.argsort()[-N:][::-1]
    sim_texts_test.append([train["content"].iloc[j] for j in most_similar_indices])
    sim_measur_test.append([similarities[j] for j in most_similar_indices])
    sim_techniques_test.append([train["techniques"].iloc[j] for j in most_similar_indices])
    trigger_phrases_test.append([train["trigger_words_phrases"].iloc[j] for j in most_similar_indices])
#
# The same but text_vectors instead of triger_phrases_vectors:
sim_texts_test_t, sim_measur_test_t, sim_techniques_test_t, trigger_phrases_test_t = [], [], [], []
for i in tqdm(range(len(test))):
    sample_vector = text_vectors_test[i]
    similarities = cosine_similarity(sample_vector.reshape(1, -1), text_vectors)[0]
    for j in long_texts_ids:
        similarities[j] = 0
    most_similar_indices = similarities.argsort()[-N:][::-1]
    sim_texts_test_t.append([train["content"].iloc[j] for j in most_similar_indices])
    sim_measur_test_t.append([similarities[j] for j in most_similar_indices])
    sim_techniques_test_t.append([train["techniques"].iloc[j] for j in most_similar_indices])
    trigger_phrases_test_t.append([train["trigger_words_phrases"].iloc[j] for j in most_similar_indices])


100%|██████████| 5735/5735 [00:19<00:00, 289.35it/s]


## Construct prompts

In [222]:
EXPLANATIONS_WITH_EXAMPLES = """
- **loaded_language**: Emotionally charged words (positive/negative) to sway opinions. Example: “Packing Ukrainians into minibuses” evokes outrage.
- **glittering_generalities**: Vague, positive concepts (e.g., “freedom,” “unity”) to inspire emotion without substance. Example: “We are united! Our brave heroes will liberate Ukraine!”
- **euphoria:** Highlighting successes to create excitement and boost morale. Example: “Our forces crushed the enemy—total victory!”
- **appeal_to_fear**: Exaggerating threats to pressure action. Example: “Unregistered Ukrainians abroad will lose banking access.”
- **fud**: Spreading doubt or vague threats to destabilize trust. Example: “Was Zelensky really in Bakhmut? No one can know for sure.”
- **bandwagon**: Urging alignment with a (claimed) popular trend. Example: “Sanction skepticism is rising in Germany—join the movement.”
- **cliche**: Overused phrases to shut down debate. Example: “We’ll never know the full truth” or “Focus on winning, not details.”
- **whataboutism**: Deflecting criticism by accusing the other side of hypocrisy. Example: “But your side did the same thing—why is it different now?”
- **cherry_picking**: Selectively presenting facts to support a biased narrative. Example: Highlighting regions failing mobilization while ignoring successes.
- **straw_man**: Misrepresenting an argument to easily attack it. Example: “Gender-sensitive education means corrupting children.”
"""

EXPLANATIONS_WITHOUT_EXAMPLES = """
- **loaded_language**: Emotionally charged words (positive/negative) to sway opinions.
- **glittering_generalities**: Vague, positive concepts (e.g., “freedom,” “unity”) to inspire emotion without substance.
- **euphoria**: Highlighting successes to create excitement and boost morale.
- **appeal_to_fear**: Exaggerating threats to pressure action.
- **fud**: Spreading doubt or vague threats to destabilize trust.
- **bandwagon**: Urging alignment with a (claimed) popular trend.
- **cliche**: Overused phrases to shut down debate.
- **whataboutism**: Deflecting criticism by accusing the other side of hypocrisy.
- **cherry_picking**: Selectively presenting facts to support a biased narrative.
- **straw_man**: Misrepresenting an argument to easily attack it.
"""

In [ ]:
def build_prompted_sample(sample):  
    prompt = f"""  
    <start_of_turn>system  
You are an AI trained to detect rhetorical manipulation in social media.  
Return ONLY the technique names from the list, comma-separated.<end_of_turn>  

<start_of_turn>user  
### Task: Identify techniques in this post using ONLY the following:  

{EXPLANATIONS_WITH_EXAMPLES}  

### Examples: 
    """
    Examples_strings = ["\n" + f" Example: POST: {ex_text[:500]}" + "\n" + \
                        f"TECHNIQUES: {', '.join(ex_tech) if len(ex_tech) > 0 else 'None'}" + "\n" # + \
                        # f"TRIGER PHRASES: {', '.join(ex_ph) if len(ex_ph) > 0 else 'None'}"
        for ex_text, ex_tech, ex_ph in zip(sample["examples_texts"], sample["examples_techniques"], sample['examples_trigger_phrases'])] 

    prompt += "\n".join(Examples_strings) 
    prompt += f"""
### POST to analyze: {sample["content"]}<end_of_turn>  

<start_of_turn>assistant  
### PREDICTED_OUTPUT: """  

    sample["prompt"] = prompt.strip()  
    sample["output"] = "TECHNIQUES: " + ', '.join(sample["techniques"])  #+ "\n" + \
        # "TRIGER PHRASES: " + ', '.join(sample["trigger_phrases"]) 
    return sample

samples = [
    {
        "content": train["content"].iloc[i],
        "techniques": train["techniques"].iloc[i] if train["techniques"].iloc[i] is not None else [],
        "trigger_words": train["trigger_words"].iloc[i] if train["trigger_words"].iloc[i] is not None else [],
        "trigger_phrases": train["trigger_words_phrases"].iloc[i] if train["trigger_words_phrases"].iloc[i] is not None else [],
        "examples_texts": [i if i is not None else "" for i in list(sim_texts[i]) + list(sim_texts_t[i])],
        "examples_techniques": [i if i is not None else [] for i in list(sim_techniques[i]) + list(sim_techniques_t[i])],
        "examples_trigger_phrases": [i if i is not None else [] for i in list(trigger_phrases[i]) + list(trigger_phrases_t[i])]
    }
    for i in range(len(train))
]
samples = [build_prompted_sample(sample) for sample in samples]

In [ ]:
# Save the samples to a file:
joblib.dump(samples, os.path.join(TRAIN_PATH, "samples_prompt_train.joblib"))

In [ ]:
samples = [
    {
        "content": test["content"].iloc[i],
        "techniques": [],
        "trigger_words": [],
        "examples_texts": [i if i is not None else "" for i in list(sim_texts_test[i]) + list(sim_texts_test_t[i])],
        "examples_techniques": [i if i is not None else [] for i in list(sim_techniques_test[i]) + list(sim_techniques_test_t[i])],
        "examples_trigger_phrases": [i if i is not None else [] for i in list(trigger_phrases_test[i]) + list(trigger_phrases_test_t[i])],
        "trigger_phrases": []
    }
    for i in range(len(test))
]
samples = [build_prompted_sample(sample) for sample in samples]

In [ ]:
# Save the samples to a file:
joblib.dump(samples, os.path.join(TRAIN_PATH, "samples_prompt_test.joblib"))